# Met Office MOGREPS-UK (UK Regional Ensemble) – AWS ASDI Demo

This notebook demonstrates how to use the `site_archive_aws.MOGREPSUK` accessor
to read and visualise Met Office MOGREPS-UK ensemble data from AWS S3.

## Dataset
MOGREPS-UK is the Met Office's operational UK-domain ensemble prediction system.
It produces 18-member ensemble forecasts at ~2.2 km grid spacing out to 54 hours,
with output every 3 hours.

## Requirements
```
pip install pyearthtools-archive-aws
```


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import site_archive_aws
from site_archive_aws import MOGREPSUK

print(f"site_archive_aws version: {site_archive_aws.__version__}")

## 1. Configure the Accessor

In [ ]:
QUERY_TIME = "2023-06-01T03:00"

# Load a subset of members for speed in this demo
accessor = MOGREPSUK("2t", members=list(range(6)), anon=True)
print(accessor)

## 2. Load Ensemble Data

In [ ]:
ds = accessor[QUERY_TIME]
print(ds)
print("Realization values:", ds["realization"].values)

t2m = ds["air_temperature"].squeeze(dim="time", drop=True) - 273.15
print("Shape (realization, lat, lon):", t2m.shape)

## 3. Ensemble-Mean 2-m Temperature over the UK

In [ ]:
ens_mean = t2m.mean("realization")

fig, ax = plt.subplots(
    figsize=(8, 10),
    subplot_kw={"projection": ccrs.PlateCarree()},
)

im = ax.contourf(
    ens_mean["longitude"],
    ens_mean["latitude"],
    ens_mean.values,
    levels=np.linspace(-5, 30, 36),
    cmap="RdBu_r",
    transform=ccrs.PlateCarree(),
    extend="both",
)

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.set_extent([-10, 3, 49, 61], crs=ccrs.PlateCarree())
ax.gridlines(draw_labels=True, linewidth=0.3)

plt.colorbar(im, ax=ax, label="2-m Temperature (°C)", shrink=0.7)
ax.set_title(
    f"MOGREPS-UK – Ensemble Mean 2-m Temperature\n{QUERY_TIME} (UTC)",
    fontsize=13,
)
plt.tight_layout()
plt.savefig("mogreps_uk_2t_mean.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to mogreps_uk_2t_mean.png")

## 4. Probability of Temperature > 20 °C

In [ ]:
threshold = 20.0  # °C
prob_warm = (t2m > threshold).mean("realization") * 100  # percentage

fig, ax = plt.subplots(
    figsize=(8, 10),
    subplot_kw={"projection": ccrs.PlateCarree()},
)

im = ax.contourf(
    prob_warm["longitude"],
    prob_warm["latitude"],
    prob_warm.values,
    levels=np.arange(0, 110, 10),
    cmap="YlOrRd",
    transform=ccrs.PlateCarree(),
)

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.set_extent([-10, 3, 49, 61], crs=ccrs.PlateCarree())
ax.gridlines(draw_labels=True, linewidth=0.3)

plt.colorbar(im, ax=ax, label=f"P(T2m > {threshold} °C) [%]", shrink=0.7)
ax.set_title(
    f"MOGREPS-UK – P(2-m T > {threshold} °C)\n{QUERY_TIME} (UTC)",
    fontsize=13,
)
plt.tight_layout()
plt.savefig("mogreps_uk_prob_warm.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to mogreps_uk_prob_warm.png")

## 5. Summary

In this notebook we:
1. Loaded MOGREPS-UK 2-m temperature for 6 ensemble members.
2. Plotted the ensemble-mean temperature over the UK.
3. Computed and displayed the probability of temperatures exceeding 20 °C.

See `scripts/demo_mogreps_uk.py` for the command-line equivalent.